In [0]:
df = spark.table("workspace.caio_s07.retail_transactions_course_dataset")

In [0]:
#workspace.caio_s07.retail_transactions_course_dataset

In [0]:
display(df)

MINI LAB 02

In [0]:
df_clean = df

In [0]:
df_clean= df_clean.dropDuplicates(["transaction_id"])

In [0]:
df_clean= df_clean.dropna(subset=["transaction_id", "transaction_ts", "total_eur"])

In [0]:
display(df_clean)

In [0]:
df_clean.count()

In [0]:
df_clean.describe()

In [0]:
df_clean.printSchema()

In [0]:
df.count()
df_clean.count()
original_count = df.count()
cleaned_count = df_clean.count()
print(f"Original count: {original_count}")
print(f"Cleaned count: {cleaned_count}")

Delta Lake

In [0]:
df_clean.write.format("delta").mode("overwrite").saveAsTable("retail_delta")

In [0]:
%sql
SELECT * FROM retail_delta

In [0]:
%sql
UPDATE retail_delta
SET discount_pct = '0'
WHERE discount_pct = 'N/A';

In [0]:
%sql
SELECT *
FROM retail_delta VERSION AS OF 0
LIMIT 5;

In [0]:
%sql
SELECT * 
FROM retail_delta
LIMIT 5;

In [0]:
%sql
SELECT *
FROM retail_delta VERSION AS OF 0
LIMIT 5;

In [0]:
%sql
DESCRIBE HISTORY retail_delta;

(
  SELECT * FROM retail_delta VERSION AS OF 0
EXCEPT
SELECT * FROM retail_delta VERSION AS OF 1
)
UNION ALL 
(
  SELECT * FROM retail_delta VERSION AS OF 1
  EXCEPT
  SELECT * FROM retail_delta VERSION AS OF 0
);

In [0]:
%sql

DESCRIBE HISTORY retail_delta;

(
  SELECT * FROM retail_delta VERSION AS OF 0
EXCEPT
SELECT * FROM retail_delta VERSION AS OF 1
)
UNION ALL 
(
  SELECT * FROM retail_delta VERSION AS OF 1
  EXCEPT
  SELECT * FROM retail_delta VERSION AS OF 0
);
DESCRIBE HISTORY retail_delta;

MINI LAB 03: DATA LIVE TABLE

In [0]:
from pyspark.sql.functions import coalesce, lit

df = spark.read.table("workspace.caio_s07.retail_transactions_course_dataset")
df_updated = df.withColumn("promo_code", coalesce("promo_code", lit("No hay promo code")))
display(df_updated)

In [0]:
spark.sql("""
UPDATE workspace.caio_s07.retail_transactions_course_dataset
SET promo_code = 'No hay promo code'
WHERE promo_code IS NULL
""")